# SOC Analyst environment in Google Colab

This notebook installs the [Soc Analyst](https://huggingface.co/spaces/Suryaai05/Soc-Analyst-Final-Repo) FastAPI app, runs it in the background, and calls the API from Colab (localhost works **inside the same runtime**).

**Runtime:** use **GPU** if you plan PPO training (optional). First startup can take **a few minutes** (PyTorch, transformers, OpenEnv import).

**Repo:** set `REPO_URL` to your GitHub or Hugging Face Git URL (public clone).

**Data:** an optional cell loads [Cybersecurity Threat Detection Logs (Kaggle)](https://www.kaggle.com/datasets/aryan208/cybersecurity-threat-detection-logs) into the app’s log store (needs Kaggle API credentials in Colab).

In [ ]:
# --- Configure clone URL (pick one) ---
REPO_URL = "https://huggingface.co/spaces/Suryaai05/Soc-Analyst-Final-Repo"  # or "https://github.com/Ruthvik4257/Soc-Analyst-Final-Repo.git"
REPO_DIR = "/content/soc-analyst-env"

import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).is_dir():
  subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
  print("Repo already present; delete", REPO_DIR, "to re-clone fresh.")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Install dependencies (Colab has torch often; this aligns the rest with requirements.txt)
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Expose app root on PYTHONPATH and start Uvicorn in the background
import os
import subprocess
import time
import sys
from pathlib import Path

root = str(Path(os.getcwd()).resolve())
os.environ["PYTHONPATH"] = root + os.pathsep + os.environ.get("PYTHONPATH", "")
PORT = "8000"

logf = open("/tmp/uvicorn_soc.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", PORT],
    cwd=root,
    stdout=logf,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
print("Uvicorn PID:", proc.pid, "- logs: /tmp/uvicorn_soc.log")
BASE = f"http://127.0.0.1:{PORT}"

# Wait for /healthz (import can take 2–5+ minutes on CPU)
import urllib.request, json
for i in range(120):
    time.sleep(3)
    try:
        with urllib.request.urlopen(BASE + "/healthz", timeout=5) as r:
            body = r.read().decode()
            print("API up:", body[:200])
            break
    except Exception as e:
        if i == 0 or (i + 1) % 5 == 0:
            print(f"... waiting ({i+1}/120):", type(e).__name__)
else:
    print("Server did not become ready. Last part of log:")
    try:
        with open("/tmp/uvicorn_soc.log") as _lf:
            _t = _lf.read()
            print(_t[-4000:] if len(_t) > 4000 else _t)
    except OSError as _e:
        print(_e)

## (Optional) Load the Kaggle “Cybersecurity Threat Detection Logs” dataset

Use **[Cybersecurity Threat Detection Logs](https://www.kaggle.com/datasets/aryan208/cybersecurity-threat-detection-logs)** in the in-memory log store (same data the `/api/datasets/logs/*` routes use).

**Kaggle API token (pick one):**

1. **Colab Secrets (recommended):** In Kaggle: [Create API token](https://www.kaggle.com/settings) and download `kaggle.json` (or copy username + key). In Colab: **Secrets** (key icon) → add **`KAGGLE_USERNAME`** and **`KAGGLE_KEY`**. For **each** secret, turn on **Notebook access** (otherwise `userdata` cannot read them). Re-run the code cell after saving secrets.
2. **Or** upload the JSON file to **`/content/kaggle.json`** in the file sidebar (or put it on Drive as `/content/drive/MyDrive/kaggle.json` and mount Drive first).
3. **Or** (recommended for a persistent local clone) copy `notebooks/kaggle_secrets.json.example` → `notebooks/kaggle_secrets.json`, fill in username + key, and **never** commit that file (it is in `.gitignore`). In Colab, upload the same JSON to `/content/kaggle_secrets.json` or `/content/kaggle.json`.
4. **Or** at the top of the next code cell, set the optional `MANUAL_KAGGLE_USERNAME` / `MANUAL_KAGGLE_KEY` (private runtime only; don’t commit values).

The cell below runs **after** the Uvicorn cell so `BASE` is defined. Very large CSVs are capped by the server’s `MAX_LOG_ENTRIES` (default 500k rows); raise it before starting the server if you need more.

In [ ]:
# Download aryan208/cybersecurity-threat-detection-logs and POST CSV(s) to the local API
import json
import os
import sys
import subprocess
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle", "requests"])
import requests

# Optional: paste from kaggle.com/settings if you don't use Colab Secrets (clear before sharing!)
MANUAL_KAGGLE_USERNAME = ""
MANUAL_KAGGLE_KEY = ""

def _colab_secret(name: str) -> str:
    """Colab SecretNotFoundError if missing — resolve one key at a time."""
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        return (v or "").strip()
    except Exception:
        return ""

def _resolve_kaggle_creds() -> tuple[str, str]:
    u = (os.environ.get("KAGGLE_USERNAME") or MANUAL_KAGGLE_USERNAME or "").strip()
    k = (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_TOKEN") or MANUAL_KAGGLE_KEY or "").strip()
    if not u:
        u = _colab_secret("KAGGLE_USERNAME")
    if not k:
        k = _colab_secret("KAGGLE_KEY") or _colab_secret("KAGGLE_TOKEN")
    return u, k

# --- Kaggle credentials → ~/.kaggle/kaggle.json ---
KAGGLE_DIR = Path("/root/.kaggle")
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
cfg = KAGGLE_DIR / "kaggle.json"
if not cfg.is_file():
    u, k = _resolve_kaggle_creds()
    if u and k:
        cfg.write_text(json.dumps({"username": u, "key": k}))
        os.chmod(cfg, 0o600)

_CANDIDATES = (
    Path("notebooks/kaggle_secrets.json"),  # gitignored; same format as kaggle.json
    Path("/content/kaggle_secrets.json"),
    Path("/content/kaggle.json"),
    Path("/content/drive/MyDrive/kaggle.json"),  # after mounting Google Drive
)
if not cfg.is_file():
    import shutil
    for src in _CANDIDATES:
        if src.is_file():
            shutil.copy2(src, cfg)
            os.chmod(cfg, 0o600)
            break
if not cfg.is_file():
    _u, _k = _resolve_kaggle_creds()
    raise RuntimeError(
        "Kaggle API credentials not found. Do one of:\n"
        "  • Colab Secrets: add KAGGLE_USERNAME and KAGGLE_KEY from kaggle.com/settings, "
        'and enable **Notebook access** for each.\n'
        f"  • Upload kaggle.json to: {[str(p) for p in _CANDIDATES]}\n"
        "  • Or set MANUAL_KAGGLE_USERNAME / MANUAL_KAGGLE_KEY in this cell.\n"
        f"  • Diagnostics: username set={bool(_u)}, key set={bool(_k)}."
    )

DATASET = "aryan208/cybersecurity-threat-detection-logs"
OUT = Path("/content/kaggle_aryan208_cyber")
OUT.mkdir(parents=True, exist_ok=True)

from kaggle.api.kaggle_api_extended import KaggleApi  # noqa: E402

api = KaggleApi()
api.authenticate()
api.dataset_download_files(DATASET, path=str(OUT), unzip=True, quiet=False)

csvs = sorted(OUT.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
if not csvs:
    raise FileNotFoundError(f"No .csv under {OUT}; list the dataset on Kaggle and adjust paths.")

# Largest CSV only; use `UPLOAD_CSVS = csvs` to import every .csv in the download
UPLOAD_CSVS = csvs[:1]

# Optional: clear previous uploads so this run is the only source
_ = requests.post(f"{BASE}/api/datasets/logs/clear", timeout=60)
print("clear:", _.json())

for csv_path in UPLOAD_CSVS:
    print("Uploading", csv_path, "—", csv_path.stat().st_size, "bytes")
    with open(csv_path, "rb") as f:
        r = requests.post(
            f"{BASE}/api/datasets/logs/upload",
            files={"file": (csv_path.name, f, "text/csv")},
            timeout=600,
        )
    r.raise_for_status()
    print(r.json())

print("GET summary:", requests.get(f"{BASE}/api/datasets/logs/summary", timeout=30).json())

## Call the API from this notebook
If you ran the **Kaggle import above**, `/api/datasets/logs/summary` will show your imported [Cybersecurity Threat Detection Logs](https://www.kaggle.com/datasets/aryan208/cybersecurity-threat-detection-logs) rows; otherwise the count is zero until you upload logs.

These requests go to **127.0.0.1** in the same Colab VM (no tunnel needed).
Open **`/docs`** in the optional tunnel below if you want a browser on another machine.

In [ ]:
import json
import urllib.request

def get(path: str) -> str:
    with urllib.request.urlopen(BASE + path) as r:
        return r.read().decode()

def post_json(path: str, data: dict) -> dict:
    body = json.dumps(data).encode("utf-8")
    req = urllib.request.Request(BASE + path, data=body, headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read().decode())

print(get("/api/datasets/logs/summary"))  # also: /api/datasets/summary (alias in current server)
print(post_json("/api/train", {
    "episodes": 2,
    "model_name": "distilgpt2",
    "learning_rate": 1e-5,
    "push_to_hub": False,
    "mode": "single_agent",
    "campaign_length": 20,
    "negotiation_rounds": 2,
    "seed": 42,
}))

## Optional: public URL (ngrok)
1. Get a free token at [ngrok.com](https://ngrok.com/) → **Authtoken**.
2. Colab: **Secrets** (key icon) add `NGROK_AUTH_TOKEN`.
3. Run the cell below to open the web UI in a new tab. **Do not** share tokens.

In [ ]:
# Optional public HTTPS URL to the same Uvicorn port (for browser: /, /docs)
# Run the Uvicorn cell first; port must match (default 8000).
import os
import subprocess
import sys

PUBLIC_PORT = 8000
try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    token = os.environ.get("NGROK_AUTH_TOKEN", "")
if not token:
    print("Set Secret NGROK_AUTH_TOKEN or env NGROK_AUTH_TOKEN to use ngrok.")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
    from pyngrok import ngrok  # type: ignore
    ngrok.set_auth_token(token)
    http_tunnel = ngrok.connect(PUBLIC_PORT, bind_tls=True)
    public = str(http_tunnel.public_url)
    print("Public URL:", public)
    print("API docs:", public + "/docs")